# BDA
## Notebook interactivo para ejecutar consultas SQL
## El resultado se carga como un pandas DataFrame

---

## Instalación de dependencias

In [1]:
# Instalar dependencias necesarias
%pip install pandas mysql-connector-python psycopg2-binary ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 9.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 8.4 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 9.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 12.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [ipywidgets]7 [pandas]onnector-python]
Note: you may need to restart the kernel to use updated packages.


##  Imports

In [2]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
# pandas avisa de que solo prueba oficialmente conexiones SQLAlchemy; con sqlite3,
# mysql.connector y psycopg2 funciona igual, así que solo se oculta ese aviso
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy', category=UserWarning)

print(" Librerías importadas correctamente")

 Librerías importadas correctamente


##  Configuración y Conexión a la Base de Datos

Selecciona el tipo de base de datos y configura los parámetros de conexión:

In [ ]:
# Variables globales para almacenar la conexión
current_connection = None
connection_type = None
is_connected = False

# Widgets de configuración
db_type = widgets.Dropdown(
    options=['SQLite', 'MariaDB/MySQL', 'PostgreSQL'],
    value='SQLite',
    description='Tipo de BD:',
    style={'description_width': '120px'}
)

# Widgets para SQLite
sqlite_file = widgets.Text(
    value='./docencia.db',
    description='Archivo DB:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Widgets para MariaDB/PostgreSQL
host = widgets.Text(
    value='localhost',
    description='Host:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

port = widgets.IntText(
    value=3306,
    description='Puerto:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

database = widgets.Text(
    value='docencia',
    description='Base de datos:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

user = widgets.Text(
    value='masteruser',
    description='Usuario:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

password = widgets.Password(
    value='',
    description='Contraseña:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Widget para esquema de PostgreSQL
schema = widgets.Text(
    value='docencia',
    description='Esquema:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Botón de conexión
connect_btn = widgets.Button(
    description=' Conectar',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)

disconnect_btn = widgets.Button(
    description=' Desconectar',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='40px'),
    disabled=True
)

# Output para mensajes de conexión
connection_output = widgets.Output()

# Contenedor para campos de conexión
connection_box = widgets.VBox()

def update_connection_fields(change):
    """Actualiza los campos según el tipo de BD seleccionado"""
    if change['new'] == 'SQLite':
        connection_box.children = [sqlite_file]
    elif change['new'] == 'MariaDB/MySQL':
        port.value = 3306
        database.value = 'docencia'
        connection_box.children = [host, port, database, user, password]
    elif change['new'] == 'PostgreSQL':
        port.value = 5432
        database.value = 'master'
        connection_box.children = [host, port, database, user, password, schema]

def on_connect_clicked(b):
    """Maneja la conexión a la base de datos"""
    global current_connection, connection_type, is_connected
    
    with connection_output:
        clear_output()
        print("   Conectando a la base de datos...\n")
        
        try:
            if db_type.value == 'SQLite':
                import sqlite3
                current_connection = sqlite3.connect(sqlite_file.value)
                connection_type = 'sqlite'
                print(f"   Conectado a SQLite: {sqlite_file.value}")
                
            elif db_type.value == 'MariaDB/MySQL':
                import mysql.connector
                current_connection = mysql.connector.connect(
                    host=host.value,
                    port=port.value,
                    database=database.value,
                    user=user.value,
                    password=password.value
                )
                connection_type = 'mariadb'
                print(f"   Conectado a MariaDB/MySQL")
                print(f"   Host: {host.value}:{port.value}")
                print(f"   Base de datos: {database.value}")
                
            elif db_type.value == 'PostgreSQL':
                import psycopg2
                current_connection = psycopg2.connect(
                    host=host.value,
                    port=port.value,
                    database=database.value,
                    user=user.value,
                    password=password.value,
                    options=f'-c search_path={schema.value}'
                )
                connection_type = 'postgres'
                print(f"   Conectado a PostgreSQL")
                print(f"   Host: {host.value}:{port.value}")
                print(f"   Base de datos: {database.value}")
                print(f"   Esquema: {schema.value}")
            
            is_connected = True
            connect_btn.disabled = True
            disconnect_btn.disabled = False
            db_type.disabled = True
            print("\nAhora puedes ejecutar consultas SQL")
            
        except Exception as e:
            print(f"ERROR al conectar: {e}")
            is_connected = False

def on_disconnect_clicked(b):
    """Maneja la desconexión de la base de datos"""
    global current_connection, connection_type, is_connected
    
    with connection_output:
        clear_output()
        try:
            if current_connection is not None:
                current_connection.close()
                print(f"Desconectado de {db_type.value}")
            current_connection = None
            connection_type = None
            is_connected = False
            connect_btn.disabled = False
            disconnect_btn.disabled = True
            db_type.disabled = False
        except Exception as e:
            print(f"Error al desconectar: {e}")

# Configurar el comportamiento inicial
db_type.observe(update_connection_fields, names='value')
update_connection_fields({'new': db_type.value})

# Vincular eventos
connect_btn.on_click(on_connect_clicked)
disconnect_btn.on_click(on_disconnect_clicked)

# Mostrar interfaz de conexión
display(HTML("<h3> Configuración de Conexión</h3>"))
display(db_type)
display(connection_box)
display(HTML("<br>"))
display(widgets.HBox([connect_btn, disconnect_btn]))
display(connection_output)

Dropdown(description='Tipo de BD:', options=('SQLite', 'MariaDB/MySQL', 'PostgreSQL'), style=DescriptionStyle(…

Output()

##  Ejecutar Consultas SQL

Una vez conectado, puedes ejecutar consultas SQL:

In [ ]:
# Variable global para resultados
df_resultado = None

# Área de consulta SQL (más grande)
sql_query = widgets.Textarea(
    value='SELECT * FROM tabla',
    description='Consulta SQL:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='900px', height='250px')
)

# Botón de ejecución
execute_btn = widgets.Button(
    description=' Ejecutar Consulta',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

# Botón de exportación
export_btn = widgets.Button(
    description=' Exportar TSV',
    button_style='info',
    layout=widgets.Layout(width='200px', height='40px')
)

# Nombre del archivo TSV
export_filename = widgets.Text(
    value='resultados.tsv',
    description='Archivo:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

# Output para resultados
query_output = widgets.Output()

def on_execute_clicked(b):
    """Maneja la ejecución de consultas SQL"""
    global df_resultado, current_connection, connection_type
    
    with query_output:
        clear_output()
        
        if not is_connected or current_connection is None:
            print("   ERROR: No hay conexión activa")
            print("   Por favor, conecta primero a una base de datos")
            return
        
        print(" Ejecutando consulta...\n")
        print("="*70)
        
        try:
            # Para PostgreSQL, establecer el esquema antes de la consulta
            if connection_type == 'postgres':
                cursor = current_connection.cursor()
                cursor.execute(f"SET search_path TO {schema.value}")
                cursor.close()
            
            # Ejecutar consulta
            df_resultado = pd.read_sql(sql_query.value, current_connection)
            
            # Mostrar resultados
            total_filas = len(df_resultado)
            print(f"\n TOTAL DE FILAS: {total_filas}")
            print("="*70)
            
            print(f"\n  INFORMACIÓN DEL DATAFRAME:")
            print(f"   - Columnas: {list(df_resultado.columns)}")
            print(f"   - Dimensiones: {df_resultado.shape}")
            print(f"   - Tipos de datos:")
            for col, dtype in df_resultado.dtypes.items():
                print(f"      • {col}: {dtype}")
            
            print(f"\n PRIMERAS 20 FILAS:")
            print("="*70)
            display(df_resultado.head(20))
            
            if df_resultado.select_dtypes(include=['number']).shape[1] > 0:
                print(f"\n ESTADÍSTICAS DESCRIPTIVAS:")
                print("="*70)
                display(df_resultado.describe())
            
            print(f"\n Consulta ejecutada exitosamente")
            print(f" Los datos están disponibles en la variable 'df_resultado'")
            
        except Exception as e:
            print(f" ERROR al ejecutar consulta: {e}")
            df_resultado = None

def on_export_clicked(b):
    """Maneja la exportación a TSV"""
    with query_output:
        if df_resultado is not None:
            try:
                filename = export_filename.value
                df_resultado.to_csv(filename, sep='\t', index=False)
                print(f"\n Datos exportados exitosamente a: {filename}")
                print(f"   - Formato: TSV (Tab-Separated Values)")
                print(f"   - Filas exportadas: {len(df_resultado)}")
                print(f"   - Columnas exportadas: {len(df_resultado.columns)}")
            except Exception as e:
                print(f"\n Error al exportar: {e}")
        else:
            print("\n No hay resultados para exportar. Ejecuta primero una consulta.")

# Vincular eventos
execute_btn.on_click(on_execute_clicked)
export_btn.on_click(on_export_clicked)

# Mostrar interfaz de consultas
display(HTML("<h3> Consulta SQL</h3>"))
display(sql_query)
display(widgets.HBox([execute_btn, export_btn]))
display(HTML("<br><h3> Exportación</h3>"))
display(export_filename)
display(query_output)

## Trabajar con los Resultados

Puedes manipular el DataFrame `df_resultado` en esta celda:

In [ ]:
# Verificar si hay resultados disponibles
if df_resultado is not None:
    print(f"DataFrame cargado con {len(df_resultado)} filas\n")
    
    # df_resultado.head()
    # df_resultado.info()
    # df_resultado['columna'].value_counts()
    # df_resultado.groupby('columna').sum()
    
else:
    print(" No hay resultados disponibles. Ejecuta primero una consulta.")